# M2 — Segmentación de Instancias de Vehículos (YOLO11-seg)
Entrena las **5 clases obligatorias**: `moto`, `auto`, `minibus`, `camion`, `bus`.

In [ ]:
!nvidia-smi -L || echo "Activa GPU T4 en Entorno de ejecucion > Cambiar tipo de entorno"
!pip -q install -U ultralytics==8.4.218 roboflow==1.7.4 pyyaml==6.0.4 pandas matplotlib requests
# CLIP solo se instala si se reconstruyen mascaras con YOLOE desde cero
# !pip -q install 'git+https://github.com/ultralytics/CLIP.git'

In [ ]:
import os, glob, json, shutil, random, yaml, cv2, requests, numpy as np, torch
from pathlib import Path
from getpass import getpass
from roboflow import Roboflow
from ultralytics import YOLO, YOLOE

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
WORKSPACE = 'cristopher-diego-rodriguez-ayala'
PROJECT_M2 = 'peaje-vehiculos-seg-6laub'
USAR_DATASET_ROBOFLOW_PREPARADO = True  # Descarga directo la version ya curada y aumentada en Roboflow

CLASES_OBJETIVO = ['moto', 'auto', 'minibus', 'camion', 'bus']
ID_OBJETIVO = {c: i for i, c in enumerate(CLASES_OBJETIVO)}
MAPA_SINONIMOS = {
    'moto': 'moto', 'motorcycle': 'moto', 'motorbike': 'moto', 'motociclista': 'moto', 'scooter': 'moto',
    'auto': 'auto', 'car': 'auto', 'sedan': 'auto', 'suv': 'auto', 'pickup': 'auto', 'taxi': 'auto',
    'minibus': 'minibus', 'minibús': 'minibus', 'micro': 'minibus', 'trufi': 'minibus', 'van': 'minibus', 'minivan': 'minibus',
    'camion': 'camion', 'camión': 'camion', 'truck': 'camion', 'cargo truck': 'camion', 'trailer': 'camion',
    'bus': 'bus', 'autobus': 'bus', 'flota': 'bus', 'buses': 'bus',
}

ROBOFLOW_API_KEY = os.environ.get('ROBOFLOW_API_KEY') or getpass('Pega tu Roboflow Private API Key: ')
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
print('Dispositivo:', DEVICE, '| Workspace:', WORKSPACE)

In [ ]:
BASE_DL = Path('/content/m2_fuentes')
MERGED_COCO = Path('/content/m2_coco_upload')
MERGED_YOLO = Path('/content/m2_yolo_train')
for d in (BASE_DL, MERGED_COCO, MERGED_YOLO):
    shutil.rmtree(d, ignore_errors=True)
    d.mkdir(parents=True, exist_ok=True)

def descargar_rf(ws_slug, proj_slug, ver, fmt, dest):
    try:
        ds = rf.workspace(ws_slug).project(proj_slug).version(ver).download(fmt, location=str(dest))
        print('Descargado:', proj_slug, '->', ds.location)
        return ds
    except Exception as e:
        print('Aviso descarga', proj_slug, e)
        return None

registros = []
if USAR_DATASET_ROBOFLOW_PREPARADO:
    ds_seg = ds_bo1 = ds_bo2 = None
    wiki_imgs = []
    print('Modo rapido activo: se descargara directamente la version curada desde Roboflow.')
else:
    ds_seg = descargar_rf('helmet-o8tke', 'vehicle-segmentation-pyt1u', 1, 'yolov11', BASE_DL / 'seg_base')
    ds_bo1 = descargar_rf('lab2e3', 'transporte-publico-bolivia', 3, 'yolov8', BASE_DL / 'bolivia_1')
    ds_bo2 = descargar_rf('jaqueline-paola-coronel-condori', 'transporte-publico', 1, 'yolov8', BASE_DL / 'bolivia_2')

    WIKI_DIR = BASE_DL / 'wikimedia'
    WIKI_DIR.mkdir(parents=True, exist_ok=True)
    queries_wiki = [
        ('minibus', 'Minibus La Paz Bolivia'),
        ('bus', 'Bus Bolivia flota'),
        ('camion', 'Truck Bolivia highway'),
        ('moto', 'Motorcycle street'),
        ('auto', 'Sedan car front view'),
    ]
    headers = {'User-Agent': 'IAM811-PeajeEducativo/1.0 (https://unifranz.edu.bo)'}
    wiki_imgs = []
    for clase_hint, q in queries_wiki:
        try:
            r = requests.get('https://commons.wikimedia.org/w/api.php', params={
                'action': 'query', 'generator': 'search', 'gsrsearch': q, 'gsrnamespace': 6,
                'gsrlimit': 15, 'prop': 'imageinfo', 'iiprop': 'url|mime', 'iiurlwidth': 960, 'format': 'json'
            }, headers=headers, timeout=15).json()
            for pid, pinfo in ((r.get('query') or {}).get('pages') or {}).items():
                ii = (pinfo.get('imageinfo') or [{}])[0]
                url = ii.get('thumburl') or ii.get('url')
                mime = ii.get('mime', '')
                if url and ('jpeg' in mime or 'png' in mime):
                    resp = requests.get(url, headers=headers, timeout=15)
                    if resp.status_code == 200 and len(resp.content) > 8000:
                        fp = WIKI_DIR / f'wiki_{clase_hint}_{pid}.jpg'
                        fp.write_bytes(resp.content)
                        wiki_imgs.append((fp, clase_hint))
        except Exception as e:
            print('Aviso Wikimedia', q, e)
    print('Imagenes Wikimedia descargadas:', len(wiki_imgs))

In [ ]:
registros = []
if USAR_DATASET_ROBOFLOW_PREPARADO:
    print('Modo rapido: se descargara el dataset M2 desde Roboflow.')
else:
    yoloe = YOLOE('yoloe-11s-seg.pt')
    PROMPTS_M2 = {
        'motorbike': 'moto', 'motorcycle': 'moto', 'scooty': 'moto',
        'car': 'auto', 'suv': 'auto', 'pickup': 'auto',
        'minibus': 'minibus', 'van': 'minibus',
        'truck': 'camion', 'cargo truck': 'camion',
        'bus': 'bus',
    }
    yoloe.set_classes(list(PROMPTS_M2.keys()), yoloe.get_text_pe(list(PROMPTS_M2.keys())))

    for sp in ('train', 'valid'):
        (MERGED_YOLO / sp / 'images').mkdir(parents=True, exist_ok=True)
        (MERGED_YOLO / sp / 'labels').mkdir(parents=True, exist_ok=True)
        (MERGED_COCO / sp).mkdir(parents=True, exist_ok=True)

    def cargar_names(ds_root):
        y = yaml.safe_load((Path(ds_root) / 'data.yaml').read_text())
        n = y['names']
        return {i: name for i, name in enumerate(n)} if isinstance(n, list) else {int(k): v for k, v in n.items()}

    for ds_src in (ds_seg, ds_bo1, ds_bo2):
        if ds_src is None:
            continue
        names_src = cargar_names(ds_src.location)
        for split_in in ('train', 'valid', 'test'):
            img_dir = Path(ds_src.location) / split_in / 'images'
            if not img_dir.exists():
                continue
            sp_out = 'valid' if split_in in ('valid', 'test') else 'train'
            for img_p in sorted(img_dir.glob('*')):
                lbl_p = img_p.parent.parent / 'labels' / f'{img_p.stem}.txt'
                if not lbl_p.exists():
                    continue
                polis = []
                for line in lbl_p.read_text().splitlines():
                    parts = line.strip().split()
                    if len(parts) < 7:
                        continue
                    raw_cls = str(names_src[int(float(parts[0]))]).lower()
                    if raw_cls == 'auto' and ds_src is ds_seg:
                        continue
                    norm_cls = MAPA_SINONIMOS.get(raw_cls) or PROMPTS_M2.get(raw_cls)
                    if norm_cls is None:
                        continue
                    coords = np.array([float(x) for x in parts[1:]], dtype=float).reshape(-1, 2)
                    polis.append((ID_OBJETIVO[norm_cls], coords))
                if polis:
                    registros.append((img_p, sp_out, polis))

    for i, (img_p, _) in enumerate(wiki_imgs):
        im = cv2.imread(str(img_p))
        if im is None:
            continue
        r = yoloe.predict(im, conf=0.22, imgsz=640, device=DEVICE, agnostic_nms=True, verbose=False)[0]
        polis = []
        if r.masks is not None and len(r.boxes):
            for poly, cid in zip(r.masks.xyn, r.boxes.cls.cpu().numpy().astype(int)):
                cls_name = PROMPTS_M2.get(r.names[cid])
                if cls_name and len(poly) >= 3:
                    polis.append((ID_OBJETIVO[cls_name], poly))
        if polis:
            registros.append((img_p, 'valid' if i % 5 == 0 else 'train', polis))

    print('Total de imagenes unificadas con poligonos:', len(registros))

In [ ]:
coco_splits = {
    sp: {
        'info': {'year': '2026', 'version': '1', 'description': 'IAM-811 M2'},
        'licenses': [{'id': 1, 'url': '', 'name': 'MIT'}],
        'images': [],
        'annotations': [],
        'categories': [{'id': i, 'name': c, 'supercategory': 'vehiculo'} for i, c in enumerate(CLASES_OBJETIVO)],
    }
    for sp in ('train', 'valid')
}
conteo_clases = {c: 0 for c in CLASES_OBJETIVO}

ws = rf.workspace(WORKSPACE)
proj = ws.project(PROJECT_M2)
versiones = proj.versions()

if USAR_DATASET_ROBOFLOW_PREPARADO and versiones:
    ver_id = max(int(v.version) for v in versiones)
    print('Descargando version Roboflow M2 existente:', ver_id)
    ds_rf = proj.version(ver_id).download('yolov11', location='/content/m2_roboflow_final')
    DATA_YAML_M2 = str(Path(ds_rf.location) / 'data.yaml')
else:
    ann_id = 1
    for idx, (src_p, sp, polis) in enumerate(registros):
        im = cv2.imread(str(src_p))
        if im is None:
            continue
        H, W = im.shape[:2]
        fname = f'{idx:05d}_{src_p.stem}.jpg'
        cv2.imwrite(str(MERGED_YOLO / sp / 'images' / fname), im)
        cv2.imwrite(str(MERGED_COCO / sp / fname), im)

        coco_splits[sp]['images'].append({'id': idx + 1, 'file_name': fname, 'width': W, 'height': H})
        yolo_lines = []
        for cid, poly in polis:
            poly = np.clip(poly, 0.0, 1.0)
            if len(poly) < 3:
                continue
            conteo_clases[CLASES_OBJETIVO[cid]] += 1
            yolo_lines.append(f'{cid} ' + ' '.join(f'{x:.6f} {y:.6f}' for x, y in poly))
            px = poly[:, 0] * W
            py = poly[:, 1] * H
            seg_flat = [round(float(v), 2) for pt in np.column_stack([px, py]) for v in pt]
            x1, y1, x2, y2 = float(px.min()), float(py.min()), float(px.max()), float(py.max())
            bw, bh = max(1.0, x2 - x1), max(1.0, y2 - y1)
            coco_splits[sp]['annotations'].append({
                'id': ann_id, 'image_id': idx + 1, 'category_id': cid,
                'segmentation': [seg_flat], 'area': round(bw * bh, 2),
                'bbox': [round(x1, 2), round(y1, 2), round(bw, 2), round(bh, 2)], 'iscrowd': 0,
            })
            ann_id += 1
        (MERGED_YOLO / sp / 'labels' / f'{idx:05d}_{src_p.stem}.txt').write_text('\n'.join(yolo_lines))

    for sp in ('train', 'valid'):
        (MERGED_COCO / sp / '_annotations.coco.json').write_text(json.dumps(coco_splits[sp]))

    yaml.safe_dump({
        'path': str(MERGED_YOLO), 'train': 'train/images', 'val': 'valid/images',
        'nc': len(CLASES_OBJETIVO), 'names': {i: c for i, c in enumerate(CLASES_OBJETIVO)},
    }, open(MERGED_YOLO / 'data.yaml', 'w'), sort_keys=False)
    DATA_YAML_M2 = str(MERGED_YOLO / 'data.yaml')
    try:
        ws.upload_dataset(dataset_path=str(MERGED_COCO), project_name=PROJECT_M2, project_type='instance-segmentation', num_workers=8)
        ver_id = proj.generate_version(settings={
            'preprocessing': {'auto-orient': True, 'resize': {'width': 640, 'height': 640, 'format': 'Stretch to'}},
            'augmentation': {
                'image': {'versions': 2},
                'flip': {'horizontal': True, 'vertical': False},
                'brightness': {'brighten': True, 'darken': True, 'percent': 15},
                'rotate': {'degrees': 10},
            },
        })
        ds_rf = proj.version(int(ver_id)).download('yolov11', location='/content/m2_roboflow_final')
        DATA_YAML_M2 = str(Path(ds_rf.location) / 'data.yaml')
    except Exception as exc:
        print('Aviso subida/versionado Roboflow:', exc)

In [ ]:
cfg = yaml.safe_load(Path(DATA_YAML_M2).read_text())
print('Entrenando M2 con:', DATA_YAML_M2, '| Clases:', cfg.get('names'))

modelo_m2 = YOLO('yolo11s-seg.pt')
modelo_m2.train(
    data=DATA_YAML_M2,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    device=0 if DEVICE == 'cuda' else 'cpu',
    project='/content/runs/segment',
    name='m2_vehiculos',
    exist_ok=True,
)

In [ ]:
BEST_M2 = Path('/content/runs/segment/m2_vehiculos/weights/best.pt')
val_res = YOLO(str(BEST_M2)).val(data=DATA_YAML_M2)
names_val = yaml.safe_load(Path(DATA_YAML_M2).read_text()).get('names', {})
if isinstance(names_val, list):
    names_val = {i: n for i, n in enumerate(names_val)}

ds_root_m2 = Path(DATA_YAML_M2).parent
total_imgs_m2 = sum(len(list((ds_root_m2 / sp / 'images').glob('*'))) for sp in ('train', 'valid') if (ds_root_m2 / sp / 'images').exists())
inst_por_clase = {str(v): 0 for v in names_val.values()}
for sp in ('train', 'valid'):
    for lp in (ds_root_m2 / sp / 'labels').glob('*.txt'):
        for ln in lp.read_text().splitlines():
            if ln.strip():
                cname = str(names_val.get(int(float(ln.split()[0])), ln.split()[0]))
                inst_por_clase[cname] = inst_por_clase.get(cname, 0) + 1

metricas_m2 = {
    'modelo': 'yolo11s-seg',
    'total_imagenes_dataset': total_imgs_m2,
    'instancias_por_clase': inst_por_clase,
    'box_map50': round(float(val_res.box.map50), 4),
    'box_map50_95': round(float(val_res.box.map), 4),
    'mask_map50': round(float(val_res.seg.map50), 4),
    'mask_map50_95': round(float(val_res.seg.map), 4),
    'mask_map50_por_clase': {
        str(names_val.get(i, i)): round(float(v), 4)
        for i, v in enumerate(val_res.seg.ap50)
    },
}
print('=== METRICAS M2 ===')
print(json.dumps(metricas_m2, indent=2, ensure_ascii=False))
Path('/content/resumen_metricas_m2.json').write_text(json.dumps(metricas_m2, indent=2, ensure_ascii=False))
shutil.copy(BEST_M2, '/content/best_m2_seg.pt')

import sys, platform
print('=== DIAGNOSTICO COLAB M2 ===')
print(json.dumps({
    'python': sys.version.split()[0], 'os': platform.platform(),
    'torch': torch.__version__, 'cuda': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}, indent=2))

try:
    from google.colab import files
    files.download('/content/best_m2_seg.pt')
    files.download('/content/resumen_metricas_m2.json')
except ImportError:
    print('Ejecucion fuera de Colab: artefactos guardados en /content')